# AEGIS GRC — NLP Control Gap Classifier & Risk Analysis Engine
**Academic / Research Angle:** Automated ISO/IEC 27001:2022 Control Gap Classification & Defensible Risk Calculus Using Natural Language Processing  
**Author:** GRC Engineering Portfolio (CloudScale Technologies Inc.)  
**Framework Focus:** ISO/IEC 27001:2022 Annex A (Organizational & Technological Controls)  
**Evaluation Standard:** Stratified 80/20 Train/Test Split (Target: Precision, Recall, Accuracy, F1 >= 0.80)

---
### Notebook Objectives:
1. Load and perform Exploratory Data Analysis (EDA) on 250 curated audit findings.
2. Engineer NLP features using sublinear TF-IDF (unigrams & bigrams).
3. Train and compare LinearSVC, Logistic Regression, and Multinomial Naive Bayes models.
4. Evaluate Accuracy, Precision, Recall, F1-Score, and plot an annotated $10 \times 10$ Confusion Matrix.
5. Demonstrate end-to-end GRC risk calculation: Inherent Risk ($L \times I$), Control Gap detection, and Residual Risk ($R_{res}$).


In [ ]:
import os
import sys
from pathlib import Path

# Configure environment and local paths
REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'data').exists() and (REPO_ROOT.parent / 'data').exists():
    REPO_ROOT = REPO_ROOT.parent

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns if 'seaborn' in sys.modules else None

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

print(f"Working Directory: {REPO_ROOT}")
print("All ML and Data Analysis libraries loaded successfully!")


## 1. Data Ingestion & Exploratory Data Analysis (EDA)

In [ ]:
dataset_path = REPO_ROOT / 'data' / 'training_dataset.csv'
df = pd.read_csv(dataset_path)

print(f"Total Audit Findings Loaded: {len(df)}")
print(f"Total Unique Control Classes: {df['Target_Control_ID'].nunique()}\n")

# Distribution by Control Class
class_counts = df['Target_Control_ID'].value_counts()
print("Sample Distribution per Control Class:")
print(class_counts)

# Quick Preview
df.head(5)


In [ ]:
plt.figure(figsize=(10, 4), dpi=150)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(class_counts)))
bars = plt.bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.8)
plt.title("Balanced Audit Finding Distribution (25 Samples per Class)", fontsize=12, fontweight='bold')
plt.xlabel("ISO 27001:2022 Annex A Control ID", fontsize=10, fontweight='bold')
plt.ylabel("Number of Labeled Audit Observations", fontsize=10, fontweight='bold')
plt.ylim(0, 30)
plt.grid(axis='y', linestyle=':', alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.8, int(yval), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 2. NLP Feature Engineering & Stratified Train/Test Split

In [ ]:
X = df['Finding_Text']
y = df['Target_Control_ID']
classes = sorted(y.unique().tolist())

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set Size: {len(X_train)} observations (80%)")
print(f"Test Set Size:     {len(X_test)} observations (20% holdout)")

# TF-IDF Feature Extraction (Unigrams + Bigrams)
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1,
    stop_words='english'
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Engineered TF-IDF Feature Space Dimensions: {X_train_vec.shape[1]} n-gram features")


## 3. Model Training & Benchmark Comparison

In [ ]:
models = {
    "Linear Support Vector (LinearSVC)": LinearSVC(C=1.0, random_state=42, max_iter=2000),
    "Logistic Regression (L2)": LogisticRegression(C=5.0, max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.05)
}

results = []
for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro Precision (%)": round(prec * 100, 2),
        "Macro Recall (%)": round(rec * 100, 2),
        "Macro F1-Score (%)": round(f1 * 100, 2),
        "Target Met (>=80%)": "PASS" if acc >= 0.80 and f1 >= 0.80 else "FAIL"
    })

results_df = pd.DataFrame(results)
results_df


## 4. Confusion Matrix Visualization & Detailed Classification Report

In [ ]:
best_model = models["Linear Support Vector (LinearSVC)"]
y_pred_best = best_model.predict(X_test_vec)

cm = confusion_matrix(y_test, y_pred_best, labels=classes)

plt.figure(figsize=(9, 7), dpi=150)
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title("AEGIS GRC — Test Set Confusion Matrix (LinearSVC)", fontsize=12, fontweight='bold', pad=15)
plt.colorbar(fraction=0.046, pad=0.04)

tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation=45, fontsize=9)
plt.yticks(tick_marks, classes, fontsize=9)
plt.xlabel("Predicted Annex A Control ID", fontsize=10, fontweight='bold')
plt.ylabel("True Labeled Control ID", fontsize=10, fontweight='bold')

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        val = cm[i, j]
        color = "white" if val > thresh else "black"
        fontweight = "bold" if val > 0 and i == j else "normal"
        plt.text(j, i, format(val, 'd'), ha="center", va="center", color=color, fontweight=fontweight, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
print("--- Detailed Per-Class Classification Report (LinearSVC) ---")
print(classification_report(y_test, y_pred_best, target_names=classes, digits=4))


## 5. End-to-End GRC Assessment Pipeline (Inherent Risk, Control Gaps, Residual Risk & Playbooks)

In [ ]:
CONTROL_PLAYBOOKS = {
    "A.5.18": ("Quarterly User Access Review & SCIM Deprovisioning", "Enforce automated SCIM deprovisioning from HRIS and quarterly manager access certifications."),
    "A.5.19": ("Third-Party Vendor Risk Assessment", "Enforce mandatory annual SOC 2 Type II collection and signed Data Processing Addenda (DPA)."),
    "A.8.1":  ("Endpoint Fleet Management & Disk Encryption", "Deploy MDM configuration profile enforcing FileVault/BitLocker full-disk encryption and screen timeout."),
    "A.8.5":  ("Phishing-Resistant MFA Enforcement", "Enforce FIDO2/WebAuthn hardware-backed multi-factor authentication and deprecate SMS OTP."),
    "A.8.8":  ("Technical Vulnerability Patch Management", "Configure automated CI/CD container scanning (Trivy) and enforce 14-day SLA for Critical CVEs."),
    "A.8.12": ("Data Leakage Prevention & S3 Hardening", "Enable AWS S3 Block Public Access, enforce KMS customer-managed key encryption, and deploy DLP agents."),
    "A.8.13": ("Information Backup & Disaster Recovery", "Enable cross-region automated backup replication, immutable Object Lock, and conduct bi-annual restoration drills."),
    "A.8.20": ("Network Segmentation & Firewall Hardening", "Restrict security groups removing 0.0.0.0/0 ingress on administrative ports (22, 5432) and deploy WAF."),
    "A.8.24": ("Cryptographic Key Management & TLS 1.3", "Enforce TLS 1.3 in transit and AES-256 KMS customer-managed key encryption at rest."),
    "A.8.28": ("Secure Coding & SAST Build Gating", "Integrate automated static analysis (SAST) and secret scanning pre-commit hooks into GitHub Actions.")
}

def evaluate_audit_finding(finding_text, current_status="Missing", likelihood=4, impact=4):
    """
    End-to-End GRC Assessment Engine:
    1. NLP Classification of ISO 27001 Annex A Control
    2. Inherent Risk Calculation: L x I
    3. Control Gap Assessment: Major Gap / Partial Gap / Verified
    4. Residual Risk Calculation: Round(Inherent x (1 - E))
    5. Tailored Remediation Playbook Assignment
    """
    # 1. NLP Inference
    vec_text = vectorizer.transform([finding_text])
    pred_control = best_model.predict(vec_text)[0]
    
    # 2. Inherent Risk
    inherent_risk = likelihood * impact
    
    # 3. Control Gap & Effectiveness
    status_clean = current_status.strip().title()
    if status_clean in ["Missing", "Failed", "Open", "Disabled", "Major Gap"]:
        gap_type = "Major Gap"
        eff = 0.00
    elif status_clean in ["Partial", "Inconsistent", "In Progress", "Partial Gap"]:
        gap_type = "Partial Gap"
        eff = 0.35
    else:
        gap_type = "Verified"
        eff = 0.75
        
    # 4. Residual Risk
    # 4. Residual Risk (Preventive controls mitigate likelihood by up to 65%, detective controls mitigate impact by up to 45%)
    res_l = max(1, round(likelihood * (1.0 - (eff * 0.65))))
    res_i = max(1, round(impact * (1.0 - (eff * 0.45))))
    residual_risk = res_l * res_i
    res_tier = "Critical" if residual_risk >= 20 else ("High" if residual_risk >= 12 else ("Medium" if residual_risk >= 6 else "Low"))
    
    # 5. Playbook
    playbook_title, playbook_action = CONTROL_PLAYBOOKS.get(
        pred_control, ("General Control Hardening", "Review and remediate according to ISMS guidelines.")
    )
    
    return {
        "Observation": finding_text,
        "Predicted_Control": pred_control,
        "Inherent_Risk": f"{inherent_risk} (L={likelihood}, I={impact})",
        "Control_Gap": gap_type,
        "Residual_Risk": f"{residual_risk} ({res_tier}, L={res_l}, I={res_i})",
        "Remediation_Playbook": playbook_title,
        "Engineering_Action": playbook_action
    }

# Test on 3 real-world scenarios
samples = [
    ("Production web server running OpenSSL library vulnerable to unpatched Critical CVE-2023-3817.", "Missing", 4, 5),
    ("AWS S3 customer document storage bucket configured with public read access enabled to the internet.", "Open", 4, 5),
    ("Engineering VPN login relies on legacy SMS one-time passcodes without WebAuthn MFA.", "In Progress", 3, 4)
]

sample_results = [evaluate_audit_finding(s[0], s[1], s[2], s[3]) for s in samples]
pd.DataFrame(sample_results)
